# MONAI Lung CT Registration - Data Ingestion

Download the paired lung CT dataset from Zenodo and upload to Snowflake stages.

## Dataset
- **Source**: [Paired Lung CT Dataset](https://zenodo.org/record/3835682)
- **Format**: NIfTI (.nii.gz) 3D volumes
- **Contents**: 10 cases with inspiration/expiration CT pairs and lung masks

## Prerequisites
- Run `01_setup.sql` first

In [7]:
!pip install -q snowflake-ml-python

In [24]:
import os
import zipfile
import tempfile
import urllib.request

from snowflake.snowpark import Session

## Configuration

In [25]:
# Connection name from ~/.snowflake/connections.toml
# See: https://docs.snowflake.com/en/developer-guide/snowpark/python/creating-session
CONNECTION_NAME = "monai-quickstart"  # Change to match your connection name

# Snowflake objects created by 01_setup.sql
ROLE = "MONAI_USER"
WAREHOUSE = "MONAI_WH"
DATABASE = "MONAI_QUICKSTART_DB"
SCHEMA = "ML"
STAGE = "@MEDICAL_IMAGES_STG"

# Dataset URL
ZENODO_URL = "https://zenodo.org/record/3835682/files/training.zip"

In [26]:
# Create session using connection from connections.toml
session = Session.builder.config("connection_name", CONNECTION_NAME).create()

# Switch to MONAI resources
session.use_role(ROLE)
session.use_warehouse(WAREHOUSE)
session.use_database(DATABASE)
session.use_schema(SCHEMA)

print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

Connected: "MONAI_QUICKSTART_DB"."ML"


## Download Dataset

In [27]:
temp_dir = tempfile.mkdtemp()
zip_path = os.path.join(temp_dir, "training.zip")

print(f"Downloading from {ZENODO_URL}...")
urllib.request.urlretrieve(ZENODO_URL, zip_path)
print(f"Downloaded: {os.path.getsize(zip_path) / 1e6:.1f} MB")

# Extract
extract_dir = os.path.join(temp_dir, "training")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(temp_dir)

print(f"Extracted to: {extract_dir}")

Downloaded: 279.0 MB
Extracted to: /var/folders/wh/1t1qbkgx1csgf4q8fxkt5gnr0000gn/T/tmp0ixga558/training


## Upload to Snowflake Stage

In [31]:
# Dataset structure:
# training/
#   scans/      - CT scans (both expiration and inspiration)
#   lungMasks/  - lung segmentation masks

folders = ["scans", "lungMasks"]
total_files = 0

for folder in folders:
    folder_path = os.path.join(extract_dir, folder)
    if os.path.exists(folder_path):
        files = [f for f in os.listdir(folder_path) if f.endswith('.nii.gz')]
        print(f"Uploading {folder}/: {len(files)} files...")
        
        for f in files:
            local_path = os.path.join(folder_path, f)
            session.file.put(local_path, f"{STAGE}/{folder}/", auto_compress=False, overwrite=True)
        
        total_files += len(files)

print(f"Uploaded {total_files} files to {STAGE}")

Uploading scans/: 40 files...
Uploading lungMasks/: 40 files...
Uploaded 80 files to @MEDICAL_IMAGES_STG


## Verify Upload

In [32]:
files_df = session.sql(f"LIST {STAGE}").to_pandas()
print(f"Files in stage: {len(files_df)}")
print(files_df.head(10))

Files in stage: 80
                                              "name"  "size"  \
0   medical_images_stg/lungMasks/case_001_exp.nii.gz   60000   
1  medical_images_stg/lungMasks/case_001_insp.nii.gz   91520   
2   medical_images_stg/lungMasks/case_002_exp.nii.gz   98736   
3  medical_images_stg/lungMasks/case_002_insp.nii.gz   92656   
4   medical_images_stg/lungMasks/case_003_exp.nii.gz   72640   
5  medical_images_stg/lungMasks/case_003_insp.nii.gz   88016   
6   medical_images_stg/lungMasks/case_004_exp.nii.gz   73680   
7  medical_images_stg/lungMasks/case_004_insp.nii.gz   72304   
8   medical_images_stg/lungMasks/case_005_exp.nii.gz   84928   
9  medical_images_stg/lungMasks/case_005_insp.nii.gz   86512   

                              "md5"                "last_modified"  
0  fae7c3528dd8db6b512a0572d19887a2  Tue, 24 Feb 2026 02:24:15 GMT  
1  b2dfed9bac2b9aa32ec81225ddcdc3be  Tue, 24 Feb 2026 02:24:17 GMT  
2  3b0c2395ea82970ff75fa8b272ff6265  Tue, 24 Feb 2026 02:24:14 GMT  


In [33]:
# Cleanup local temp files
import shutil
shutil.rmtree(temp_dir, ignore_errors=True)

session.close()
print("Done! Proceed to 03_train_and_register.ipynb")

Done! Proceed to 03_train_and_register.ipynb
